In [17]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

1. Load Dataset

In [18]:
df = pd.read_csv('dataset/train.csv')

print("="*60)
print("TITANIC DATASET - FEATURE ENGINEERING & SELECTION")
print("="*60)
print(f"Original dataset shape: {df.shape}")
print(f"Original features: {df.columns.tolist()}")

TITANIC DATASET - FEATURE ENGINEERING & SELECTION
Original dataset shape: (891, 12)
Original features: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


2. Feature Engineering

In [19]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Dr': 5, 'Rev': 5, 'Col': 5, 'Major': 5, 'Mlle': 2, 'Countess': 3, 'Ms': 2, 'Lady': 3, 'Jonkheer': 1, 'Don': 1, 'Dona': 3, 'Mme': 3, 'Capt': 5, 'Sir': 1}
df['Title'] = df['Title'].map(title_mapping).fillna(5)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=[1, 2, 3, 4, 5]).astype(float)
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

# New Features Stats
print(f"\nNew features summary:")
new_features = ['FamilySize', 'IsAlone', 'Title', 'AgeGroup', 'FarePerPerson']
for feature in new_features:
    print(f"- {feature}: {df[feature].nunique()} unique values, "
          f"{df[feature].isnull().sum()} missing values")

feature_columns = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'] + new_features
df_features = df[feature_columns + ['Survived']].copy()


New features summary:
- FamilySize: 9 unique values, 0 missing values
- IsAlone: 2 unique values, 0 missing values
- Title: 5 unique values, 0 missing values
- AgeGroup: 5 unique values, 177 missing values
- FarePerPerson: 289 unique values, 0 missing values


3. Handle missing values

In [20]:
df_features.loc[:, 'Age'] = df_features['Age'].fillna(df_features['Age'].median())
df_features.loc[:, 'AgeGroup'] = df_features['AgeGroup'].fillna(df_features['AgeGroup'].mode()[0])
df_features.loc[:, 'Fare'] = df_features['Fare'].fillna(df_features['Fare'].median())
df_features.loc[:, 'FarePerPerson'] = df_features['FarePerPerson'].fillna(df_features['FarePerPerson'].median())
df_features.loc[:, 'Embarked'] = df_features['Embarked'].fillna(df_features['Embarked'].mode()[0])

4. Encode categorical variables

In [21]:
le_sex = LabelEncoder()
df_features.loc[:, 'Sex'] = le_sex.fit_transform(df_features['Sex'])
le_embarked = LabelEncoder()
df_features.loc[:, 'Embarked'] = le_embarked.fit_transform(df_features['Embarked'])

X = df_features.drop('Survived', axis=1)
y = df_features['Survived']

5. Feature Selection Methods

In [22]:
# 1: SelectKBest
selector_kbest = SelectKBest(score_func=f_classif, k=5)
X_kbest = selector_kbest.fit_transform(X, y)

selected_features_kbest = X.columns[selector_kbest.get_support()].tolist()
feature_scores = selector_kbest.scores_

print("Top 5 features selected by SelectKBest:")
kbest_results = []
for i, (feature, score) in enumerate(zip(selected_features_kbest, 
                                        feature_scores[selector_kbest.get_support()]), 1):
    print(f"{i}. {feature:<15}: F-score = {score:.2f}")
    kbest_results.append((feature, score))


# 2. RFE

rf_estimator = RandomForestClassifier(n_estimators=100, random_state=42)
selector_rfe = RFE(estimator=rf_estimator, n_features_to_select=5, step=1)
X_rfe = selector_rfe.fit_transform(X, y)

selected_features_rfe = X.columns[selector_rfe.get_support()].tolist()
feature_rankings = selector_rfe.ranking_

print("Top 5 features selected by RFE:")
for i, feature in enumerate(selected_features_rfe, 1):
    print(f"{i}. {feature}")

print(f"\nFeature rankings (1=selected, higher=eliminated earlier):")
for feature, rank in zip(X.columns, feature_rankings):
    status = "SELECTED" if rank == 1 else f"Rank {rank}"
    print(f"{feature:<15}: {status}")

Top 5 features selected by SelectKBest:
1. Pclass         : F-score = 115.03
2. Sex            : F-score = 372.41
3. Fare           : F-score = 63.03
4. Title          : F-score = 184.40
5. FarePerPerson  : F-score = 45.91
Top 5 features selected by RFE:
1. Sex
2. Age
3. Fare
4. Title
5. FarePerPerson

Feature rankings (1=selected, higher=eliminated earlier):
Pclass         : Rank 2
Sex            : SELECTED
Age            : SELECTED
SibSp          : Rank 4
Parch          : Rank 7
Fare           : SELECTED
Embarked       : Rank 6
FamilySize     : Rank 3
IsAlone        : Rank 8
Title          : SELECTED
AgeGroup       : Rank 5
FarePerPerson  : SELECTED


6. Model Performance Comparison

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Performance with all features
rf_all = RandomForestClassifier(n_estimators=100, random_state=42)
rf_all.fit(X_train, y_train)
y_pred_all = rf_all.predict(X_test)
accuracy_all = accuracy_score(y_test, rf_all.predict(X_test))

# Performance with SelectKBest features
X_train_kbest = selector_kbest.transform(X_train)
X_test_kbest = selector_kbest.transform(X_test)
rf_kbest = RandomForestClassifier(n_estimators=100, random_state=42)
rf_kbest.fit(X_train_kbest, y_train)
y_pred_kbest = rf_kbest.predict(X_test_kbest)
accuracy_kbest = accuracy_score(y_test, y_pred_kbest)

# Performance with RFE features
X_train_rfe = selector_rfe.transform(X_train)
X_test_rfe = selector_rfe.transform(X_test)
rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42)
rf_rfe.fit(X_train_rfe, y_train)
y_pred_rfe = rf_rfe.predict(X_test_rfe)
accuracy_rfe = accuracy_score(y_test, y_pred_rfe)


print(f"Accuracy with all features ({len(X.columns)}): {accuracy_all:.4f}")
print(f"Accuracy with SelectKBest (5):   {accuracy_kbest:.4f} ({accuracy_kbest-accuracy_all:+.4f})")
print(f"Accuracy with RFE (5):           {accuracy_rfe:.4f} ({accuracy_rfe-accuracy_all:+.4f})")

Accuracy with all features (12): 0.8101
Accuracy with SelectKBest (5):   0.7877 (-0.0223)
Accuracy with RFE (5):           0.8212 (+0.0112)


7. Feature Importance Analysis

In [24]:
print("\n" + "="*50)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*50)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_all.feature_importances_
}).sort_values('importance', ascending=False)

print("All features ranked by Random Forest importance:")
for i, (_, row) in enumerate(feature_importance.iterrows(), 1):
    marker = "*" if row['feature'] in selected_features_rfe else " "
    print(f"{i:2d}. {marker} {row['feature']:<15}: {row['importance']:.4f}")

print("\n* = Selected by RFE")


FEATURE IMPORTANCE ANALYSIS
All features ranked by Random Forest importance:
 1. * Age            : 0.1665
 2. * Sex            : 0.1640
 3. * FarePerPerson  : 0.1569
 4. * Title          : 0.1568
 5. * Fare           : 0.1552
 6.   Pclass         : 0.0558
 7.   FamilySize     : 0.0391
 8.   AgeGroup       : 0.0316
 9.   Embarked       : 0.0252
10.   SibSp          : 0.0242
11.   Parch          : 0.0156
12.   IsAlone        : 0.0091

* = Selected by RFE


8. Print detailed results

In [25]:
print("\n" + "="*50)
print("ENGINEERED FEATURES IMPACT")
print("="*50)

engineered_in_top5 = [f for f in new_features if f in selected_features_rfe or f in selected_features_kbest]
print(f"Engineered features in top 5 selections: {len(engineered_in_top5)}")
for feature in engineered_in_top5:
    methods = []
    if feature in selected_features_kbest:
        methods.append("SelectKBest")
    if feature in selected_features_rfe:
        methods.append("RFE")
    print(f"- {feature}: Selected by {', '.join(methods)}")

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Original features:                    {len(feature_columns) - len(new_features)}")
print(f"Engineered features:                  {len(new_features)}")
print(f"Total features after engineering:     {len(X.columns)}")
print(f"Selected features (top 5):            5")
print(f"Best performing method:               {'SelectKBest' if accuracy_kbest >= accuracy_rfe else 'RFE'}")
print(f"Best accuracy improvement:            {max(accuracy_kbest, accuracy_rfe) - accuracy_all:+.4f}")
print(f"Engineered features in final selection: {len(engineered_in_top5)}")

# Show overlap between methods
overlap = set(selected_features_kbest) & set(selected_features_rfe)
print(f"Features selected by both methods:    {len(overlap)} ({list(overlap)})")


ENGINEERED FEATURES IMPACT
Engineered features in top 5 selections: 2
- Title: Selected by SelectKBest, RFE
- FarePerPerson: Selected by SelectKBest, RFE

FINAL SUMMARY
Original features:                    7
Engineered features:                  5
Total features after engineering:     12
Selected features (top 5):            5
Best performing method:               RFE
Best accuracy improvement:            +0.0112
Engineered features in final selection: 2
Features selected by both methods:    4 (['Fare', 'Title', 'Sex', 'FarePerPerson'])
